In [2]:
# Cell 1: Load libraries and preprocessed UAV data

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    classification_report
)

import json
import time

# Path where preprocessed files were saved in preprocess_uav.ipynb
save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical/processed/"

# Load numpy arrays (faster than CSV for large arrays)
X_train = np.load(save_path + "X_train.npy")
X_test = np.load(save_path + "X_test.npy")

# Load labels (squeeze converts DataFrame to 1D array)
y_train = pd.read_csv(save_path + "y_train.csv").squeeze()
y_test = pd.read_csv(save_path + "y_test.csv").squeeze()

# Load class names (needed for evaluatin report)
le_classes = pd.read_csv(save_path + "label_classes.csv").squeeze().tolist()

# Load feature names (needed for SHAP and LIME later)
feature_names= pd.read_csv(save_path + "feature_names.csv").squeeze().tolist()

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"Classes: {le_classes}")
print("\nData loaded!")

X_train: (23171, 37)
X_test: (9931, 37)
Classes: ['DoS attack', 'Replay', 'benign']

Data loaded!


In [3]:
# Cell 2: Train Random Forest classifier
# n_estimators = 100 means 100 decision trees in the forest
# random_state = 42 ensures same results every run (reproducibility)
# n_jobs = -1 uses all CPU cores to speed up training 

print("Training Random Forest...")
start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100, # 100 trees in the forest
    random_state=42, # fixed seed for reproducibility
    n_jobs=-1 # use all CPU cores
)

# fit = train the model on training data
rf_model.fit(X_train, y_train)

end_time = time.time()
rf_train_time = round(end_time - start_time, 2)
print(f"Training complete!")
print(f"Training time: {rf_train_time} seconds")

Training Random Forest...
Training complete!
Training time: 3.17 seconds


In [4]:
# Cell 3: Evaluate Random Forest on test data
# predict() = use trained model to make predictions on new data
print("Evaluating Random Forest...")

start_time = time.time()
y_pred = rf_model.predict(X_test)
end_time = time.time()
rf_pred_time = round(end_time - start_time, 2)

# accuracy = overcall correct predictions / total predictions
acc = accuracy_score(y_test, y_pred)

# precision = of all predicted attacks, how many were real attacks
# weighted = accounts for class imbalance

p = precision_score(y_test, y_pred, average='weighted')

# recall = of all real attacks, how many did we catch
r = recall_score(y_test, y_pred, average='weighted')

# F1 = balanced score combining precision and recall
f1 = f1_score(y_test, y_pred, average='weighted')

print("\n===Random Forest Results (UAV dataset) ===")
print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {p:.4f} ({p*100:.2f}%)")
print(f"Recal: {r:.4f} ({r*100:.2f}%)")
print(f"F1_score: {f1:.4f} ({f1*100:.2f}%)")
print(f"\nDetailed Report:")
print(classification_report(y_test, y_pred, target_names=le_classes))


Evaluating Random Forest...

===Random Forest Results (UAV dataset) ===
Accuracy: 0.9925 (99.25%)
Precision: 0.9925 (99.25%)
Recal: 0.9925 (99.25%)
F1_score: 0.9925 (99.25%)

Detailed Report:
              precision    recall  f1-score   support

  DoS attack       0.99      0.99      0.99      3501
      Replay       0.99      0.99      0.99      3602
      benign       1.00      1.00      1.00      2828

    accuracy                           0.99      9931
   macro avg       0.99      0.99      0.99      9931
weighted avg       0.99      0.99      0.99      9931



In [5]:
# Cell 4: Save model and results
import joblib

# save trained RF model
# joblib is best for saving sklearn models
joblib.dump(rf_model, save_path + "rf_model.joblib")
print("RF model saved!")

# Save results as JSON
# JSON = machine-readable format for benchmark comparison
rf_results = {
    "model": "RandomForest",
    "dataset": "UAV-Cyber-Physical",
    "accuracy": round(float(acc), 4),
    "precision": round(float(p), 4),
    "recall":  round(float(r), 4),
    "f1_score": round(float(f1), 4),
    "training_time": rf_train_time,
    "prediction_time": rf_pred_time,
    "n_estimators": 100,
    "classes": le_classes
}

with open(save_path + "rf_results.json", "w") as f:
    json.dump(rf_results, f, indent=4)
print("RF results JSON saved!")

print(f"\nSummary:")
print(f" Accuracy: {acc*100:.2f}%")
print(f" F1_score: {f1*100:.2f}%")
print(f" Train time: {rf_train_time}s")


RF model saved!
RF results JSON saved!

Summary:
 Accuracy: 99.25%
 F1_score: 99.25%
 Train time: 3.17s
